# Fresh Retail: Starter Notebook

This notebook accompanies the **Introduction** slide deck (`FreshRetail_Introduction.pptx`). It provides a shared data pipeline for both tracks, then produces the exact outputs previewed in the showcase slides.

**Run sections 1–4 first** (shared setup), then run the section for your track:

| Section | Track | What you produce |
|---------|-------|-----------------|
| 5. Operations | Ops | Temporal profiles, heatmaps, KPIs, hourly patterns |
| 6. Data Science | DS | WAPE baselines, forecast overlays, demand recovery, error analysis |

Both tracks use the same dataset, same helper functions, and same time split.

- **Operations Track**: O1 (Diagnosis) or O2 (Decision)
- **Data Science Track**: D1 (Direct benchmark) or D2 (Recovery first)

**Dataset**: [Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)

---
## 1. Setup and Data Download

In [ ]:
# Run this cell on Google Colab (already installed locally)
!pip install -q pandas pyarrow matplotlib seaborn datasets

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Setup complete.")

Setup complete.


In [3]:
from datasets import load_dataset

print("Downloading FreshRetailNet-50K from Hugging Face...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print(ds)

# Convert to pandas
train_raw = ds["train"].to_pandas()
eval_raw = ds["eval"].to_pandas()

print(f"\nTrain: {train_raw.shape}, Eval: {eval_raw.shape}")
print(f"Columns: {list(train_raw.columns)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 4500000
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 350000
    })
})

Train: (4500000, 19), Eval: (350000, 19)
Columns: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'sto

---
## 2. Data Preparation

In [4]:
def prepare_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the raw HF dataset into a clean analysis panel."""
    df = df.copy()

    # Parse date
    df["dt"] = pd.to_datetime(df["dt"])
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

    # Create series_id (unique store x product combination)
    series_keys = df[["store_id", "product_id"]].drop_duplicates().reset_index(drop=True)
    series_keys["series_id"] = range(1, len(series_keys) + 1)
    df = df.merge(series_keys, on=["store_id", "product_id"], how="left")

    # Create day index (days since start)
    min_date = df["dt"].min()
    df["day_idx"] = (df["dt"] - min_date).dt.days + 1

    n_series = df["series_id"].nunique()
    n_days = df["day_idx"].nunique()
    print(f"Prepared {len(df):,} rows \u2014 {n_series:,} series x {n_days} days")
    print(f"Date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
    return df


history = prepare_panel(train_raw)
history.head()

Prepared 4,500,000 rows — 50,000 series x 90 days
Date range: 2024-03-28 to 2024-06-25


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,...,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level,series_id,day_idx
0,0,0,2,29,78,82,4,2024-03-28,0.5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, ...",...,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ...",0.882,0,1,1.6999,15.48,73.54,1.97,1,1
1,0,0,2,29,78,82,4,2024-03-29,1.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,0,1,3.0190,15.08,76.56,1.71,1,2
2,0,0,2,29,78,82,4,2024-03-30,5.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,1,1,2.0942,15.91,76.47,1.73,1,3
3,0,0,2,29,78,82,4,2024-03-31,4.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.879,1,1,1.5618,16.13,77.40,1.76,1,4
4,0,0,2,29,78,82,4,2024-04-01,0.7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,0,1,3.5386,15.37,78.26,1.25,1,5


---
## 3. Shared Functions: flag_censoring, make_features, time_split

In [5]:
def flag_censoring(df: pd.DataFrame) -> pd.DataFrame:
    """Add censoring flags based on stockout hours."""
    df = df.copy()
    df["is_censored"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
    df["censoring_severity"] = df["stock_hour6_22_cnt"] / 16
    print(f"Censored rows: {df['is_censored'].sum():,} / {len(df):,} ({df['is_censored'].mean():.1%})")
    return df


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lag and rolling features for EDA and forecasting."""
    df = df.sort_values(["series_id", "day_idx"]).copy()
    grp = df.groupby("series_id")["sale_amount"]
    df["sales_lag1"] = grp.shift(1)
    df["sales_lag7"] = grp.shift(7)
    df["sales_roll7"] = grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df["sales_roll28"] = grp.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df["psd"] = grp.transform("mean")  # per-series daily mean
    return df


def time_split(df: pd.DataFrame, horizon: int = 7) -> tuple:
    """Split into train and validation by time. Validation = last `horizon` days."""
    min_day = df["day_idx"].min()
    max_day = df["day_idx"].max()
    val_start = max_day - horizon + 1
    train = df[df["day_idx"] < val_start].copy()
    val = df[df["day_idx"] >= val_start].copy()
    print(f"Train: day {min_day}..{val_start - 1} ({len(train):,} rows), Val: day {val_start}..{max_day} ({len(val):,} rows)")
    return train, val

In [6]:
# Apply shared pipeline
history = flag_censoring(history)
history = make_features(history)

train, val = time_split(history, horizon=7)
print(f"\nValidation window: day {val['day_idx'].min()} to {val['day_idx'].max()}")

Censored rows: 1,992,006 / 4,500,000 (44.3%)
Train: day 1..83 (4,150,000 rows), Val: day 84..90 (350,000 rows)

Validation window: day 84 to 90


---
## 4. Data at a Glance

In [ ]:
# Show a real series with stockouts
series_stockouts = history.groupby("series_id")["is_censored"].mean()
example_sid = series_stockouts[(series_stockouts > 0.3) & (series_stockouts < 0.7)].index[0]

s_example = history[history["series_id"] == example_sid][
    ["dt", "day_idx", "sale_amount", "stock_hour6_22_cnt", "is_censored", "discount", "holiday_flag", "avg_temperature"]
].head(14)
print(f"Series {example_sid} \u2014 first 14 days (a product with frequent stockouts):")
display(s_example)

In [ ]:
# Dataset dimensions
summary = pd.Series({
    "Total rows": f"{len(history):,}",
    "Series (store x product)": f"{history['series_id'].nunique():,}",
    "Days per series": str(history["day_idx"].nunique()),
    "Products (product_id)": str(history["product_id"].nunique()),
    "Stores (store_id)": str(history["store_id"].nunique()),
    "Cities (city_id)": str(history["city_id"].nunique()),
    "Management groups": str(history["management_group_id"].nunique()),
    "Mean daily sales": f"{history['sale_amount'].mean():.3f}",
    "Censored rows": f"{history['is_censored'].sum():,} ({history['is_censored'].mean():.1%})",
    "Low-sale series (psd<1)": f"{(history.groupby('series_id')['psd'].first() < 1).sum():,}",
    "High-sale series (psd>=1)": f"{(history.groupby('series_id')['psd'].first() >= 1).sum():,}",
})
display(summary.to_frame("Value"))

In [ ]:
# Sales distribution and per-series daily mean
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clipped = history["sale_amount"].clip(upper=history["sale_amount"].quantile(0.99))
axes[0].hist(clipped, bins=50, color="#065A82", edgecolor="white")
axes[0].set_title("Distribution of daily sales (clipped at 99th pctl)")
axes[0].set_xlabel("sale_amount")
axes[0].set_ylabel("Count")

psd_vals = history.groupby("series_id")["psd"].first()
axes[1].hist(psd_vals, bins=50, color="#1C7293", edgecolor="white")
axes[1].axvline(1.0, color="#E74C3C", linestyle="--", linewidth=2, label="psd=1 cutoff")
axes[1].set_title("Per-series daily mean (psd) distribution")
axes[1].set_xlabel("psd")
axes[1].set_ylabel("Number of series")
axes[1].legend()

plt.tight_layout()
plt.show()



## 6. Data Science Track

> **You may skip this section if you are focusing on the Operations track.**

This section produces the following outputs — each corresponds to a slide in the deck:

| Output | What it shows | Slide |
|--------|--------------|-------|
| WAPE results table (D1) | Baseline comparison: global mean vs seasonal naive vs rolling 28d | Slide 18 |
| Forecast overlay chart | Predicted vs actual for one series across the validation window | Slide 18 |
| Recovery comparison table (D2) | WAPE on raw vs corrected target — does imputation help? | Slide 19 |
| WAPE by management group | Which product groups are hardest to forecast? | Slide 20 |
| Residual histogram + error scatter | Where the model fails and why | Slide 20 |

**A strong data science project** starts from these baselines and improves on them with better features, better imputation, or a more sophisticated model — always measured by WAPE on the same time split.

### 6a. WAPE Evaluation Function

In [7]:
def compute_wape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Weighted Absolute Percentage Error."""
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - predicted)) / denom


def evaluate_forecast(val_df: pd.DataFrame, pred_col: str = "prediction") -> dict:
    """Compute WAPE overall, low-sale, high-sale, and harmonic mean.
    Only evaluates rows where stock_hour6_22_cnt == 0 (uncensored in validation)."""
    scored = val_df[val_df["stock_hour6_22_cnt"] == 0].copy()
    if len(scored) == 0:
        return {"wape_overall": np.nan}

    y = scored["sale_amount"].values
    yhat = scored[pred_col].values

    wape_all = compute_wape(y, yhat)

    low = scored[scored["psd"] < 1]
    high = scored[scored["psd"] >= 1]

    wape_low = compute_wape(low["sale_amount"].values, low[pred_col].values) if len(low) > 0 else np.nan
    wape_high = compute_wape(high["sale_amount"].values, high[pred_col].values) if len(high) > 0 else np.nan

    if np.isnan(wape_low) or np.isnan(wape_high) or wape_all == 0 or wape_low == 0 or wape_high == 0:
        hm = np.nan
    else:
        hm = 3 / (1/wape_all + 1/wape_low + 1/wape_high)

    return {
        "wape_overall": round(wape_all, 4) if not np.isnan(wape_all) else np.nan,
        "wape_low_sale": round(wape_low, 4) if not np.isnan(wape_low) else np.nan,
        "wape_high_sale": round(wape_high, 4) if not np.isnan(wape_high) else np.nan,
        "harmonic_mean": round(hm, 4) if not np.isnan(hm) else np.nan,
        "scored_rows": len(scored),
    }

print("Evaluation function ready.")

Evaluation function ready.


### 6b. D1 \u2014 Direct Benchmark: Naive Baselines on Raw Sales

In [ ]:
# --- Baseline 1: Global mean ---
series_mean = train.groupby("series_id")["sale_amount"].mean().rename("pred_global_mean")
val = val.drop(columns=["pred_global_mean", "pred_seasonal_naive", "pred_roll28", "forecast_day"], errors="ignore")
val = val.merge(series_mean, on="series_id", how="left")

# --- Baseline 2: Seasonal naive (last-week repeat) ---
val_start = val["day_idx"].min()
last_week = history[history["day_idx"].between(val_start - 7, val_start - 1)][["series_id", "day_idx", "sale_amount"]].copy()
last_week["forecast_day"] = last_week["day_idx"] + 7
last_week = last_week.rename(columns={"sale_amount": "pred_seasonal_naive"})

val = val.merge(last_week[["series_id", "forecast_day", "pred_seasonal_naive"]],
                left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left")
val = val.drop(columns=["forecast_day"], errors="ignore")
val["pred_seasonal_naive"] = val["pred_seasonal_naive"].fillna(val["pred_global_mean"])

# --- Baseline 3: Rolling 28-day mean ---
roll28 = train.groupby("series_id")["sale_amount"].apply(
    lambda x: x.tail(28).mean(), include_groups=False
).rename("pred_roll28")
val = val.merge(roll28, on="series_id", how="left")

# Evaluate all three
results = {}
for method, col in [("Global mean", "pred_global_mean"), ("Seasonal naive", "pred_seasonal_naive"), ("Rolling 28d", "pred_roll28")]:
    val["prediction"] = val[col].clip(lower=0)
    results[method] = evaluate_forecast(val)

results_df = pd.DataFrame(results).T
print("=== D1 Benchmark Results ===")
display(results_df)

In [ ]:
# --- Visualize: forecast overlay for one series ---
example_sid3 = history.groupby("series_id")["psd"].first().sort_values(ascending=False).index[5]
ex = history[history["series_id"] == example_sid3].copy()
ex_val = val[val["series_id"] == example_sid3].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ex["day_idx"], ex["sale_amount"], color="#065A82", linewidth=1.5, label="Actual (train)")
ax.plot(ex_val["day_idx"], ex_val["sale_amount"], color="#065A82", linewidth=2, linestyle="-", label="Actual (val)")
ax.plot(ex_val["day_idx"], ex_val["pred_global_mean"], color="#E67E22", linewidth=1.5, linestyle="--", label="Global mean")
ax.plot(ex_val["day_idx"], ex_val["pred_seasonal_naive"], color="#8E44AD", linewidth=1.5, linestyle="--", label="Seasonal naive")
ax.plot(ex_val["day_idx"], ex_val["pred_roll28"], color="#27AE60", linewidth=1.5, linestyle="--", label="Rolling 28d")

ax.axvline(ex_val["day_idx"].min() - 0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"Series {example_sid3}: Forecast overlay (validation window)")
ax.set_xlabel("day_idx")
ax.set_ylabel("sale_amount")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6c. D2 \u2014 Recovery First: Impute Censored Hours, Then Forecast

In [ ]:
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mark censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")

In [ ]:
# --- Simple recovery: random pool sampling ---
visible_sum = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)

imputed = op_sales_masked.copy()
imputed_count = 0
for h in range(16):
    col = imputed[:, h]
    mask = np.isnan(col)
    n_miss = mask.sum()
    if n_miss > 0:
        pool = col[~mask]
        imputed[mask, h] = np.maximum(0, rng.choice(pool, size=n_miss, replace=True))
        imputed_count += n_miss

# Rebuild corrected daily target
recovered_sum = np.nansum(imputed, axis=1)
outside_slice = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum, 0)
recovered_daily = outside_slice + recovered_sum

history["recovered_daily_sales"] = recovered_daily

print(f"Imputed {imputed_count:,} hourly cells")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales: {history['recovered_daily_sales'].mean():.4f}")

In [ ]:
# Re-split with recovered target
train_r, val_r = time_split(history, horizon=7)

# Seasonal naive on recovered target
val_start_r = val_r["day_idx"].min()
last_week_r = history[history["day_idx"].between(val_start_r - 7, val_start_r - 1)][
    ["series_id", "day_idx", "recovered_daily_sales"]
].copy()
last_week_r["forecast_day"] = last_week_r["day_idx"] + 7
last_week_r = last_week_r.rename(columns={"recovered_daily_sales": "pred_recovered_naive"})

val_r = val_r.merge(
    last_week_r[["series_id", "forecast_day", "pred_recovered_naive"]],
    left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left",
)
val_r = val_r.drop(columns=["forecast_day"], errors="ignore")
fallback = train_r.groupby("series_id")["recovered_daily_sales"].mean()
val_r["pred_recovered_naive"] = val_r["pred_recovered_naive"].fillna(val_r["series_id"].map(fallback))

# Also add seasonal naive on raw for fair comparison
val_r = val_r.merge(
    val[["series_id", "day_idx", "pred_seasonal_naive"]].drop_duplicates(),
    on=["series_id", "day_idx"], how="left",
)

# Evaluate both
d2_results = {}
for method, col in [("Seasonal naive (raw)", "pred_seasonal_naive"), ("Seasonal naive (recovered)", "pred_recovered_naive")]:
    val_r["prediction"] = val_r[col].clip(lower=0)
    d2_results[method] = evaluate_forecast(val_r)

d2_df = pd.DataFrame(d2_results).T
print("=== D2 Recovery Comparison ===")
display(d2_df)

### 6d. Error Analysis

In [ ]:
# WAPE by management group
scored = val[val["stock_hour6_22_cnt"] == 0].copy()
scored["prediction"] = scored["pred_seasonal_naive"].clip(lower=0)
scored["abs_error"] = np.abs(scored["sale_amount"] - scored["prediction"])

group_wape = scored.groupby("management_group_id").apply(
    lambda g: compute_wape(g["sale_amount"].values, g["prediction"].values),
    include_groups=False
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
group_wape.plot(kind="barh", color="#1C7293", edgecolor="white", ax=ax)
ax.set_title("WAPE by management group (seasonal naive baseline)")
ax.set_xlabel("WAPE")
ax.set_ylabel("Management Group ID")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution and error vs stockout frequency
scored["residual"] = scored["sale_amount"] - scored["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scored["residual"].clip(-5, 5), bins=60, color="#065A82", edgecolor="white")
axes[0].axvline(0, color="#E74C3C", linestyle="--", linewidth=2)
axes[0].set_title("Residual distribution (seasonal naive)")
axes[0].set_xlabel("Actual - Predicted")
axes[0].set_ylabel("Count")

series_error = scored.groupby("series_id").agg(
    mean_abs_error=("abs_error", "mean"),
).reset_index()
series_error = series_error.merge(
    history.groupby("series_id")["is_censored"].mean().rename("stockout_freq"),
    on="series_id"
)
axes[1].scatter(series_error["stockout_freq"], series_error["mean_abs_error"], alpha=0.1, s=5, color="#065A82")
axes[1].set_title("Mean absolute error vs stockout frequency")
axes[1].set_xlabel("Stockout frequency")
axes[1].set_ylabel("Mean absolute error")

plt.tight_layout()
plt.show()

In [ ]:
print("Implementing Per-Series Hourly Mean recovery (leakage-free)...")

# --- Start: Necessary definitions moved from fyqpkO6BPLFc to ensure scope ---
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mask censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")
# --- End: Necessary definitions moved ---


# Determine the validation start day from the original time_split for training data
# This ensures the means are calculated only from the training period
val_start_pshm_calc = history['day_idx'].max() - 7 + 1 # horizon is 7

# Filter history to create a training-only view for mean calculation
train_history_for_pshm_calc = history[history["day_idx"] < val_start_pshm_calc].copy()

# Calculate per-series hourly means FOR TRAINING DATA ONLY
per_series_hourly_mean_data = {}

# Iterate through unique series_id present in the training data
train_series_ids = train_history_for_pshm_calc['series_id'].unique()

for series_id in train_series_ids:
    # Get the original history indices that correspond to this series_id AND are in the training set
    original_history_indices_for_series_in_train = train_history_for_pshm_calc[train_history_for_pshm_calc['series_id'] == series_id].index

    # Extract the operating window sales and stock status for this series
    # from the FULL op_sales/op_stock arrays, but only for the training period rows.
    series_op_sales_train = op_sales[original_history_indices_for_series_in_train, :]
    series_op_stock_train = op_stock[original_history_indices_for_series_in_train, :]

    # Mask sales for stockouts within this series' training data
    series_op_sales_masked_train = np.where(series_op_stock_train == 1, np.nan, series_op_sales_train)

    # Initialize hourly_means with NaN for all hours
    hourly_means = np.full(series_op_sales_masked_train.shape[1], np.nan)

    # Calculate the mean for each hour for this specific series, ignoring NaNs
    # This mean is based ONLY on training data for this series, avoiding leakage.
    # Explicitly check for all-NaN slices to prevent RuntimeWarning
    for hour_idx in range(series_op_sales_masked_train.shape[1]):
        hour_sales_data = series_op_sales_masked_train[:, hour_idx]
        if not np.all(np.isnan(hour_sales_data)):
            hourly_means[hour_idx] = np.nanmean(hour_sales_data)

    per_series_hourly_mean_data[series_id] = hourly_means

# Impute using per-series hourly mean (apply to the full masked data using means from training)
imputed_pshm = op_sales_masked.copy() # Start with the full masked data

imputed_count_pshm = 0

for i in range(len(history)): # Iterate over each series-day observation in the full history
    current_series_id = history['series_id'].iloc[i]

    # Get the pre-calculated hourly means for this series (derived from training data)
    series_hourly_means = per_series_hourly_mean_data.get(current_series_id)

    if series_hourly_means is not None: # Means exist for this series from training data
        for h in range(16): # Iterate over each hour in the operating window
            if np.isnan(imputed_pshm[i, h]):
                mean_val = series_hourly_means[h]
                if not np.isnan(mean_val): # Only impute if a mean exists for this series-hour
                    imputed_pshm[i, h] = np.maximum(0, mean_val)
                    imputed_count_pshm += 1
                else:
                    # Fallback if specific series-hour mean is NaN (e.g., this series always censored for this hour in training)
                    imputed_pshm[i, h] = 0
    else:
        # If a series_id is in history but not in train_series_ids (e.g., it only appears in validation),
        # its hourly_means won't be in per_series_hourly_mean_data. Impute with 0 in this case.
        for h in range(16):
            if np.isnan(imputed_pshm[i, h]):
                imputed_pshm[i, h] = 0


# Rebuild corrected daily target (using the imputed_pshm which now covers the full history)
# visible_sum is from the original op_sales (non-censored hours)
visible_sum_pshm = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)
recovered_sum_pshm = np.sum(imputed_pshm, axis=1) # Sum of the (now) fully imputed hourly sales

# outside_slice is sales outside operating hours, adjusted to be non-negative
outside_slice_pshm = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum_pshm, 0)
recovered_daily_pshm = outside_slice_pshm + recovered_sum_pshm

history["recovered_daily_sales_pshm"] = recovered_daily_pshm

print(f"Imputed {imputed_count_pshm:,} hourly cells using per-series hourly mean (training data only).")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales (per-series hourly mean): {history['recovered_daily_sales_pshm'].mean():.4f}")

# Re-split with recovered target (this will now use the newly updated history)
train_r_pshm, val_r_pshm = time_split(history, horizon=7)

# LGBMRegressor on per-series hourly mean recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_pshm and val_r_pshm dataframes
# (these were already created from history with 'recovered_daily_sales_pshm')
train_Lgbm_pshm = train_r_pshm.dropna()
val_Lgbm_pshm = val_r_pshm.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_pshm = "recovered_daily_sales_pshm"

model_pshm = LGBMRegressor(random_state=1)
model_pshm.fit(train_Lgbm_pshm[features_lgbm], train_Lgbm_pshm[target_lgbm_pshm])

val_Lgbm_pshm["prediction"] = model_pshm.predict(val_Lgbm_pshm[features_lgbm])
val_Lgbm_pshm["prediction"] = val_Lgbm_pshm["prediction"].clip(lower=0)

print("=== LGBMRegressor on Per-Series Hourly Mean Recovered Data (Leakage-Free) ===")
# When evaluating, 'sale_amount' should be the actual target, which is recovered_daily_sales_pshm
val_Lgbm_pshm_eval = val_Lgbm_pshm.copy()
val_Lgbm_pshm_eval["sale_amount"] = val_Lgbm_pshm_eval[target_lgbm_pshm]

pshm_lgbm_results = evaluate_forecast(val_Lgbm_pshm_eval)
print(pshm_lgbm_results)

# If all_recovery_results is not in scope, this would need re-initialization or a new comparison table
# For now, let's just display it and assume it can be merged later.


Implementing Per-Series Hourly Mean recovery (leakage-free)...
Expanding hourly data...
Operating window: 16 hours (h06-h21)
Missing hourly cells: 14,311,536 / 72,000,000 (19.9%)


---
---
## 7. Next Steps

### Operations Track

**O1 \u2014 Diagnosis First**
- Extend the heatmaps to find which store x category combinations are most fragile
- Test whether promotions (`discount < 1`) increase late-day stockouts
- Run panel regressions with fixed effects to isolate drivers

**O2 \u2014 Decision First**
- Build a simple corrected demand estimate (impute censored hours from `hours_sale`)
- Compute newsvendor order quantities under raw vs. corrected demand
- Visualize the service vs. waste trade-off curve

### Data Science Track

**D1 \u2014 Direct Benchmark**
- Try exponential smoothing or a simple LightGBM with lag features
- Analyze errors by day-of-week to detect weekly patterns
- Compare WAPE across cities to find geographic patterns

**D2 \u2014 Recovery First**
- Try per-series mean imputation instead of global pool sampling
- Compare multiple recovery strategies on the same baseline
- Focus error analysis on high-stockout series where recovery matters most

### Cross-Track Synergies
- Operations insights (which products are most fragile) can inform DS feature engineering
- DS demand recovery estimates can feed back into operations policy evaluation
- Both tracks benefit from understanding the hourly censoring structure

### 7a. D1 - Direct Forecasting

#### D1 — LightGBM with lag features

In [ ]:
# baseline with only lag feature
from lightgbm import LGBMRegressor
# drop missing value
train_Lgbm = train.dropna()
val_Lgbm = val.dropna()

features_lag = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28"
]

target = "sale_amount"

model = LGBMRegressor(random_state=1)
model.fit(train_Lgbm[features_lag], train_Lgbm[target])

val_Lgbm["prediction"] = model.predict(val_Lgbm[features_lag])
val_Lgbm["prediction"] = val_Lgbm["prediction"].clip(lower=0)

evaluate_forecast(val_Lgbm)



Note: add more features...

In [ ]:
# add other features # why these features????

features = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'


]

model = LGBMRegressor(random_state=1)
model.fit(train_Lgbm[features], train_Lgbm[target])

val_Lgbm["prediction"] = model.predict(val_Lgbm[features])
val_Lgbm["prediction"] = val_Lgbm["prediction"].clip(lower=0)

evaluate_forecast(val_Lgbm)


Note: why these feature
check important features...

####D1 — Exponential Smoothing Benchmark

Test with sample of 5000

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler # for normalization
import pandas as pd
import numpy as np

# --- DLinear Model Definition (from previous successful implementation) ---
class MovingAverage(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1)) # permute to (batch, features, sequence_length)
        x = x.permute(0, 2, 1) # permute back
        return x

class DLinear(nn.Module):
    def __init__(self, input_len, pred_len, kernel_size=25): # Changed seq_len to input_len
        super().__init__()
        self.input_len = input_len # Changed seq_len to input_len
        self.pred_len = pred_len
        self.kernel_size = kernel_size

        self.moving_avg = MovingAverage(kernel_size, stride=1)
        self.linear_seasonal = nn.Linear(input_len, pred_len) # Changed seq_len to input_len
        self.linear_trend = nn.Linear(input_len, pred_len) # Changed seq_len to input_len

        # Initialize weights (often set to identity or similar for DLinear)
        nn.init.constant_(self.linear_seasonal.weight, 1.0 / input_len) # Changed seq_len to input_len
        nn.init.constant_(self.linear_trend.weight, 1.0 / input_len) # Changed seq_len to input_len
        nn.init.constant_(self.linear_seasonal.bias, 0.0)
        nn.init.constant_(self.linear_trend.bias, 0.0)

    def forward(self, x): # x: [Batch, Input length, Channel]
        # Decompose the input
        trend = self.moving_avg(x)
        seasonal = x - trend

        # Forecast
        seasonal_output = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        trend_output = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)

        return seasonal_output + trend_output

# --- Parameters ---
# These are assumed to be globally defined from previous cells or will be defined here if not.
# For consistency, using the names and values established in the conversation.
input_len = 14 # Lookback window (encoder length)
pred_len = 7   # Forecast horizon (decoder length)
epochs = 5     # Number of training epochs per series
learning_rate = 0.001 # Can be tuned after LR finder
batch_size = 128 # Defined for LR finder, used here too.




all_forecasts_dlinear = []

# --- Sample 5000 series for training ---
print("--- DLinear Training on 5000 Sample Series ---")
sample_size = 5000

# Ensure series_id columns are integer type
history["series_id"] = history["series_id"].astype(int)
train["series_id"] = train["series_id"].astype(int)
val["series_id"] = val["series_id"].astype(int)

print(f"history['series_id'] dtype: {history['series_id'].dtype}, head: {history['series_id'].head().tolist()}")
print(f"train['series_id'] dtype: {train['series_id'].dtype}, head: {train['series_id'].head().tolist()}")
print(f"val['series_id'] dtype: {val['series_id'].dtype}, head: {val['series_id'].head().tolist()}")

all_series_ids_for_dlinear = history["series_id"].drop_duplicates().sample(sample_size, random_state=RANDOM_SEED)
print(f"all_series_ids_for_dlinear dtype: {all_series_ids_for_dlinear.dtype}, head: {all_series_ids_for_dlinear.head().tolist()}")

# Filter train and val DataFrames for the sampled series
train_dlinear_full_sample = train[train["series_id"].isin(all_series_ids_for_dlinear)].copy()
val_dlinear_full_sample = val[val["series_id"].isin(all_series_ids_for_dlinear)].copy()

print(f"Length of train_dlinear_full_sample after filtering: {len(train_dlinear_full_sample)}")
print(f"Length of val_dlinear_full_sample after filtering: {len(val_dlinear_full_sample)}")

# Sort for time series processing
train_dlinear_full_sample = train_dlinear_full_sample.sort_values(["series_id", "day_idx"])
val_dlinear_full_sample = val_dlinear_full_sample.sort_values(["series_id", "day_idx"])

print(f"Training DLinear on {len(all_series_ids_for_dlinear)} series...")

series_processed_count = 0
for series_id in all_series_ids_for_dlinear:
    train_series_df = train_dlinear_full_sample[train_dlinear_full_sample["series_id"] == series_id].copy()
    val_series_df = val_dlinear_full_sample[val_dlinear_full_sample["series_id"] == series_id].copy()

    # Skip if not enough data for training sequences or no validation data
    if len(train_series_df) < input_len + pred_len or len(val_series_df) == 0:
        continue

    # Scale data for this series (MinMaxScaler)
    scaler = MinMaxScaler()
    train_sales_scaled = scaler.fit_transform(train_series_df["sale_amount"].values.reshape(-1, 1))

    # Prepare training sequences (input X, target Y)
    train_data_x, train_data_y = [], []
    for i in range(len(train_sales_scaled) - input_len - pred_len + 1):
        train_data_x.append(train_sales_scaled[i : i + input_len])
        train_data_y.append(train_sales_scaled[i + input_len : i + input_len + pred_len])

    if not train_data_x:
        continue # Skip if no training sequences can be formed

    train_tensor_x = torch.tensor(train_data_x, dtype=torch.float32).to(device)
    train_tensor_y = torch.tensor(train_data_y, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Prepare validation input (encoder input from last `input_len` days of train data)
    val_input_sales_raw = train_series_df["sale_amount"].values[-input_len:]
    val_input_sales_scaled = scaler.transform(val_input_sales_raw.reshape(-1, 1))
    val_input_tensor = torch.tensor(val_input_sales_scaled, dtype=torch.float32).unsqueeze(0).to(device) # Add batch dimension

    model = DLinear(input_len=input_len, pred_len=pred_len).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    # Train model for this series
    model.train()
    for epoch_idx in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

    # Predict on validation data
    model.eval()
    with torch.no_grad():
        forecast_scaled = model(val_input_tensor).cpu().numpy().flatten()
        # Inverse transform to get raw sales prediction
        forecast_raw = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()

    temp_df = val_series_df.copy()
    # Ensure prediction length matches the actual validation series length (should be pred_len=7)
    temp_df["dlinear_prediction"] = forecast_raw[:len(temp_df)]
    all_forecasts_dlinear.append(temp_df)
    series_processed_count += 1

print(f"Finished DLinear forecasting for {series_processed_count} series out of {sample_size} sampled.")

if all_forecasts_dlinear:
    dlinear_predictions_df = pd.concat(all_forecasts_dlinear)

    # Merge predictions into the full validation dataframe to ensure consistent evaluation
    # Use the original `val` dataframe as the base for merging
    val_scored_dlinear_temp = val.copy()
    val_scored_dlinear_temp["store_id"] = val_scored_dlinear_temp["store_id"].astype(str)
    val_scored_dlinear_temp["product_id"] = val_scored_dlinear_temp["product_id"].astype(str)

    # Ensure dlinear_predictions_df also has string IDs for merging
    dlinear_predictions_df["store_id"] = dlinear_predictions_df["store_id"].astype(str)
    dlinear_predictions_df["product_id"] = dlinear_predictions_df["product_id"].astype(str)

    # Merge only the dlinear_prediction column
    val_scored_dlinear_temp = val_scored_dlinear_temp.merge(
        dlinear_predictions_df[["store_id", "product_id", "day_idx", "dlinear_prediction"]],
        on=["store_id", "product_id", "day_idx"],
        how="left"
    )

    # Evaluate DLinear only on rows where predictions are available (i.e., the sampled series)
    val_scored_dlinear_filtered = val_scored_dlinear_temp.dropna(subset=['dlinear_prediction']).copy()

    result_dlinear = evaluate_forecast(
        val_scored_dlinear_filtered,
        pred_col="dlinear_prediction"
    )

    print("\n--- DLinear Evaluation (5000 Sampled Series) ---")
    display(result_dlinear)
else:
    print("No series processed for DLinear, potentially due to insufficient data or sampling issues.")

Note:

####D1 — DLinear Hyperparameter Tuning

1. Sample 1000 series
2. Create pipeline
3. Tune input_len, learning_rate, epochs
4. Generalize with full dataset

In [ ]:
# --- Sample 5000 series and pipeline ----
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler # for normalization
import pandas as pd
import numpy as np

# --- DLinear Model Definition --
class MovingAverage(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1)) # permute to (batch, features, sequence_length)
        x = x.permute(0, 2, 1) # permute back
        return x

class DLinear(nn.Module):
    def __init__(self, input_len, pred_len, kernel_size=25):
        super().__init__()
        self.input_len = input_len
        self.pred_len = pred_len
        self.kernel_size = kernel_size

        self.moving_avg = MovingAverage(kernel_size, stride=1)
        self.linear_seasonal = nn.Linear(input_len, pred_len)
        self.linear_trend = nn.Linear(input_len, pred_len)

        # Initialize weights (often set to identity or similar for DLinear)
        nn.init.constant_(self.linear_seasonal.weight, 1.0 / input_len)
        nn.init.constant_(self.linear_trend.weight, 1.0 / input_len)
        nn.init.constant_(self.linear_seasonal.bias, 0.0)
        nn.init.constant_(self.linear_trend.bias, 0.0)

    def forward(self, x): # x: [Batch, Input length, Channel]
        # Decompose the input
        trend = self.moving_avg(x)
        seasonal = x - trend

        # Forecast
        seasonal_output = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        trend_output = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)

        return seasonal_output + trend_output

# --- Parameters ---
# These are assumed to be globally defined from previous cells or will be defined here if not.
# For consistency, using the names and values established in the conversation.
input_len = 14 # Lookback window (encoder length)
pred_len = 7   # Forecast horizon (decoder length)
epochs = 5     # Number of training epochs per series
learning_rate = 0.001 # Can be tuned after LR finder
batch_size = 128 # Defined for LR finder, used here too.




all_forecasts_dlinear = []

# --- Sample 5000 series for training ---
print("--- DLinear Training on 5000 Sample Series ---")
sample_size = 10000

# Ensure series_id columns are integer type
history["series_id"] = history["series_id"].astype(int)
train["series_id"] = train["series_id"].astype(int)
val["series_id"] = val["series_id"].astype(int)

print(f"history['series_id'] dtype: {history['series_id'].dtype}, head: {history['series_id'].head().tolist()}")
print(f"train['series_id'] dtype: {train['series_id'].dtype}, head: {train['series_id'].head().tolist()}")
print(f"val['series_id'] dtype: {val['series_id'].dtype}, head: {val['series_id'].head().tolist()}")

all_series_ids_for_dlinear = history["series_id"].drop_duplicates().sample(sample_size, random_state=RANDOM_SEED)
print(f"all_series_ids_for_dlinear dtype: {all_series_ids_for_dlinear.dtype}, head: {all_series_ids_for_dlinear.head().tolist()}")

# Filter train and val DataFrames for the sampled series
train_dlinear_full_sample = train[train["series_id"].isin(all_series_ids_for_dlinear)].copy()
val_dlinear_full_sample = val[val["series_id"].isin(all_series_ids_for_dlinear)].copy()

print(f"Length of train_dlinear_full_sample after filtering: {len(train_dlinear_full_sample)}")
print(f"Length of val_dlinear_full_sample after filtering: {len(val_dlinear_full_sample)}")

# Sort for time series processing
train_dlinear_full_sample = train_dlinear_full_sample.sort_values(["series_id", "day_idx"])
val_dlinear_full_sample = val_dlinear_full_sample.sort_values(["series_id", "day_idx"])

print(f"Training DLinear on {len(all_series_ids_for_dlinear)} series...")

series_processed_count = 0
for series_id in all_series_ids_for_dlinear:
    train_series_df = train_dlinear_full_sample[train_dlinear_full_sample["series_id"] == series_id].copy()
    val_series_df = val_dlinear_full_sample[val_dlinear_full_sample["series_id"] == series_id].copy()

    # Skip if not enough data for training sequences or no validation data
    if len(train_series_df) < input_len + pred_len or len(val_series_df) == 0:
        continue

    # Scale data for this series (MinMaxScaler)
    scaler = MinMaxScaler()
    train_sales_scaled = scaler.fit_transform(train_series_df["sale_amount"].values.reshape(-1, 1))

    # Prepare training sequences (input X, target Y)
    train_data_x, train_data_y = [], []
    for i in range(len(train_sales_scaled) - input_len - pred_len + 1):
        train_data_x.append(train_sales_scaled[i : i + input_len])
        train_data_y.append(train_sales_scaled[i + input_len : i + input_len + pred_len])

    if not train_data_x:
        continue # Skip if no training sequences can be formed

    train_tensor_x = torch.tensor(train_data_x, dtype=torch.float32).to(device)
    train_tensor_y = torch.tensor(train_data_y, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Prepare validation input (encoder input from last `input_len` days of train data)
    val_input_sales_raw = train_series_df["sale_amount"].values[-input_len:]
    val_input_sales_scaled = scaler.transform(val_input_sales_raw.reshape(-1, 1))
    val_input_tensor = torch.tensor(val_input_sales_scaled, dtype=torch.float32).unsqueeze(0).to(device) # Add batch dimension

    model = DLinear(input_len=input_len, pred_len=pred_len).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    # Train model for this series
    model.train()
    for epoch_idx in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

    # Predict on validation data
    model.eval()
    with torch.no_grad():
        forecast_scaled = model(val_input_tensor).cpu().numpy().flatten()
        # Inverse transform to get raw sales prediction
        forecast_raw = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()

    temp_df = val_series_df.copy()
    # Ensure prediction length matches the actual validation series length (should be pred_len=7)
    temp_df["dlinear_prediction"] = forecast_raw[:len(temp_df)]
    all_forecasts_dlinear.append(temp_df)
    series_processed_count += 1

print(f"Finished DLinear forecasting for {series_processed_count} series out of {sample_size} sampled.")

if all_forecasts_dlinear:
    dlinear_predictions_df = pd.concat(all_forecasts_dlinear)

    # Merge predictions into the full validation dataframe to ensure consistent evaluation
    # Use the original `val` dataframe as the base for merging
    val_scored_dlinear_temp = val.copy()
    val_scored_dlinear_temp["store_id"] = val_scored_dlinear_temp["store_id"].astype(str)
    val_scored_dlinear_temp["product_id"] = val_scored_dlinear_temp["product_id"].astype(str)

    # Ensure dlinear_predictions_df also has string IDs for merging
    dlinear_predictions_df["store_id"] = dlinear_predictions_df["store_id"].astype(str)
    dlinear_predictions_df["product_id"] = dlinear_predictions_df["product_id"].astype(str)

    # Merge only the dlinear_prediction column
    val_scored_dlinear_temp = val_scored_dlinear_temp.merge(
        dlinear_predictions_df[["store_id", "product_id", "day_idx", "dlinear_prediction"]],
        on=["store_id", "product_id", "day_idx"],
        how="left"
    )

    # Evaluate DLinear only on rows where predictions are available (i.e., the sampled series)
    val_scored_dlinear_filtered = val_scored_dlinear_temp.dropna(subset=['dlinear_prediction']).copy()

    result_dlinear = evaluate_forecast(
        val_scored_dlinear_filtered,
        pred_col="dlinear_prediction"
    )

    print("\n--- DLinear Evaluation (5000 Sampled Series) ---")
    display(result_dlinear)
else:
    print("No series processed for DLinear, potentially due to insufficient data or sampling issues.")

In [ ]:
# --- Tune with the learning_rates and epochs ---
# Define hyperparameter search space
learning_rates = [0.0001, 0.0005, 0.001]
epochs_to_test = [3,5,9]

# Use the same sampled series for tuning
tuning_series_ids = all_series_ids_for_dlinear

tuning_results = []

print(f"Starting DLinear hyperparameter tuning on {len(tuning_series_ids)} series...")

# Filter train and val DataFrames for the tuning sample
train_dlinear_tuning_sample = train[train["series_id"].isin(tuning_series_ids)].copy()
val_dlinear_tuning_sample = val[val["series_id"].isin(tuning_series_ids)].copy()

for lr in learning_rates:
    for ep in epochs_to_test:
        print(f"\n--- Testing LR: {lr}, Epochs: {ep} ---")
        current_all_forecasts_dlinear = []
        current_series_processed_count = 0

        for series_id in tuning_series_ids:
            train_series_df = train_dlinear_tuning_sample[train_dlinear_tuning_sample["series_id"] == series_id].copy()
            val_series_df = val_dlinear_tuning_sample[val_dlinear_tuning_sample["series_id"] == series_id].copy()

            if len(train_series_df) < input_len + pred_len or len(val_series_df) == 0:
                continue

            scaler = MinMaxScaler()
            train_sales_scaled = scaler.fit_transform(train_series_df["sale_amount"].values.reshape(-1, 1))

            train_data_x, train_data_y = [], []
            for i in range(len(train_sales_scaled) - input_len - pred_len + 1):
                train_data_x.append(train_sales_scaled[i : i + input_len])
                train_data_y.append(train_sales_scaled[i + input_len : i + input_len + pred_len])

            if not train_data_x:
                continue

            train_tensor_x = torch.tensor(train_data_x, dtype=torch.float32).to(device)
            train_tensor_y = torch.tensor(train_data_y, dtype=torch.float32).to(device)

            train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

            val_input_sales_raw = train_series_df["sale_amount"].values[-input_len:]
            val_input_sales_scaled = scaler.transform(val_input_sales_raw.reshape(-1, 1))
            val_input_tensor = torch.tensor(val_input_sales_scaled, dtype=torch.float32).unsqueeze(0).to(device)

            model = DLinear(input_len=input_len, pred_len=pred_len).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr)
            criterion = nn.MSELoss()

            model.train()
            for epoch_idx in range(ep):
                for batch_x, batch_y in train_loader:
                    optimizer.zero_grad()
                    output = model(batch_x)
                    loss = criterion(output, batch_y)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                forecast_scaled = model(val_input_tensor).cpu().numpy().flatten()
                forecast_raw = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()

            temp_df = val_series_df.copy()
            temp_df["dlinear_prediction"] = forecast_raw[:len(temp_df)]
            current_all_forecasts_dlinear.append(temp_df)
            current_series_processed_count += 1

        print(f"Processed {current_series_processed_count} series for LR={lr}, Epochs={ep}.")

        if current_all_forecasts_dlinear:
            dlinear_predictions_tuning_df = pd.concat(current_all_forecasts_dlinear)

            # Merge predictions into the original val dataframe to ensure consistent evaluation
            val_scored_tuning_temp = val.copy()
            val_scored_tuning_temp["store_id"] = val_scored_tuning_temp["store_id"].astype(str)
            val_scored_tuning_temp["product_id"] = val_scored_tuning_temp["product_id"].astype(str)
            dlinear_predictions_tuning_df["store_id"] = dlinear_predictions_tuning_df["store_id"].astype(str)
            dlinear_predictions_tuning_df["product_id"] = dlinear_predictions_tuning_df["product_id"].astype(str)

            val_scored_tuning_temp = val_scored_tuning_temp.merge(
                dlinear_predictions_tuning_df[["store_id", "product_id", "day_idx", "dlinear_prediction"]],
                on=["store_id", "product_id", "day_idx"],
                how="left"
            )

            val_scored_tuning_filtered = val_scored_tuning_temp.dropna(subset=['dlinear_prediction']).copy()

            result = evaluate_forecast(
                val_scored_tuning_filtered,
                pred_col="dlinear_prediction"
            )
            tuning_results.append({"learning_rate": lr, "epochs": ep, "wape_overall": result["wape_overall"], "scored_rows": result["scored_rows"]})
        else:
            tuning_results.append({"learning_rate": lr, "epochs": ep, "wape_overall": np.nan, "scored_rows": 0})

tuning_results_df = pd.DataFrame(tuning_results)
print("\n--- DLinear Hyperparameter Tuning Results ---")
display(tuning_results_df.sort_values(by="wape_overall"))

The `tuning_results_df` provides a clear overview of how different learning rates and epochs impact the overall WAPE. To make the trends more visible, let's visualize these results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=tuning_results_df,
    x='epochs',
    y='wape_overall',
    hue='learning_rate',
    marker='o',
    palette='viridis'
)

plt.title('DLinear WAPE vs. Epochs for Different Learning Rates')
plt.xlabel('Epochs')
plt.ylabel('Overall WAPE')
plt.xticks(epochs_to_test) # Ensure x-axis ticks match tested epochs
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Learning Rate')
plt.tight_layout()
plt.show()


Select the combination of learning rate 0.001 and the epoch 5 and generlaize in full data set

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler # for normalization
import pandas as pd
import numpy as np

# --- DLinear Model Definition (from previous successful implementation) ---
# Re-defining to ensure cell is self-contained if kernel state is reset
class MovingAverage(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1)) # permute to (batch, features, sequence_length)
        x = x.permute(0, 2, 1) # permute back
        return x

class DLinear(nn.Module):
    def __init__(self, input_len, pred_len, kernel_size=25):
        super().__init__()
        self.input_len = input_len
        self.pred_len = pred_len
        self.kernel_size = kernel_size

        self.moving_avg = MovingAverage(kernel_size, stride=1)
        self.linear_seasonal = nn.Linear(input_len, pred_len)
        self.linear_trend = nn.Linear(input_len, pred_len)

        # Initialize weights (often set to identity or similar for DLinear)
        nn.init.constant_(self.linear_seasonal.weight, 1.0 / input_len)
        nn.init.constant_(self.linear_trend.weight, 1.0 / input_len)
        nn.init.constant_(self.linear_seasonal.bias, 0.0)
        nn.init.constant_(self.linear_trend.bias, 0.0)

    def forward(self, x): # x: [Batch, Input length, Channel]
        # Decompose the input
        trend = self.moving_avg(x)
        seasonal = x - trend

        # Forecast
        seasonal_output = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        trend_output = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)

        return seasonal_output + trend_output

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Parameters (using optimal from tuning) ---
input_len = 28 # Lookback window (encoder length)
pred_len = 7   # Forecast horizon (decoder length)
learning_rate = 0.001 # Optimal learning rate from tuning
epochs = 5     # Optimal epochs from tuning
batch_size = 128

all_forecasts_dlinear_full = []

print("--- DLinear Training on ALL Series ---")

# Use ALL unique series_ids from history
all_series_ids_full = history["series_id"].drop_duplicates()

# Ensure series_id columns are integer type
history["series_id"] = history["series_id"].astype(int)
train["series_id"] = train["series_id"].astype(int)
val["series_id"] = val["series_id"].astype(int)

print(f"Processing {len(all_series_ids_full)} unique series...")

# Filter train and val DataFrames for the full set of series
train_dlinear_full_data = train[train["series_id"].isin(all_series_ids_full)].copy()
val_dlinear_full_data = val[val["series_id"].isin(all_series_ids_full)].copy()

# Sort for time series processing
train_dlinear_full_data = train_dlinear_full_data.sort_values(["series_id", "day_idx"])
val_dlinear_full_data = val_dlinear_full_data.sort_values(["series_id", "day_idx"])

series_processed_count = 0
for series_id in all_series_ids_full:
    train_series_df = train_dlinear_full_data[train_dlinear_full_data["series_id"] == series_id].copy()
    val_series_df = val_dlinear_full_data[val_dlinear_full_data["series_id"] == series_id].copy()

    # Skip if not enough data for training sequences or no validation data
    if len(train_series_df) < input_len + pred_len or len(val_series_df) == 0:
        continue

    # Scale data for this series (MinMaxScaler)
    scaler = MinMaxScaler()
    train_sales_scaled = scaler.fit_transform(train_series_df["sale_amount"].values.reshape(-1, 1))

    # Prepare training sequences (input X, target Y)
    train_data_x, train_data_y = [], []
    for i in range(len(train_sales_scaled) - input_len - pred_len + 1):
        train_data_x.append(train_sales_scaled[i : i + input_len])
        train_data_y.append(train_sales_scaled[i + input_len : i + input_len + pred_len])

    if not train_data_x:
        continue # Skip if no training sequences can be formed

    train_tensor_x = torch.tensor(train_data_x, dtype=torch.float32).to(device)
    train_tensor_y = torch.tensor(train_data_y, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Prepare validation input (encoder input from last `input_len` days of train data)
    val_input_sales_raw = train_series_df["sale_amount"].values[-input_len:]
    val_input_sales_scaled = scaler.transform(val_input_sales_raw.reshape(-1, 1))
    val_input_tensor = torch.tensor(val_input_sales_scaled, dtype=torch.float32).unsqueeze(0).to(device) # Add batch dimension

    model = DLinear(input_len=input_len, pred_len=pred_len).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    # Train model for this series
    model.train()
    for epoch_idx in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

    # Predict on validation data
    model.eval()
    with torch.no_grad():
        forecast_scaled = model(val_input_tensor).cpu().numpy().flatten()
        # Inverse transform to get raw sales prediction
        forecast_raw = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()

    temp_df = val_series_df.copy()
    # Ensure prediction length matches the actual validation series length (should be pred_len=7)
    temp_df["dlinear_prediction"] = forecast_raw[:len(temp_df)]
    all_forecasts_dlinear_full.append(temp_df)
    series_processed_count += 1

print(f"Finished DLinear forecasting for {series_processed_count} series out of {len(all_series_ids_full)} total unique series.")

if all_forecasts_dlinear_full:
    dlinear_predictions_full_df = pd.concat(all_forecasts_dlinear_full)

    # Merge predictions into the full validation dataframe to ensure consistent evaluation
    # Use the original `val` dataframe as the base for merging
    val_scored_dlinear_full_temp = val.copy()
    val_scored_dlinear_full_temp["store_id"] = val_scored_dlinear_full_temp["store_id"].astype(str)
    val_scored_dlinear_full_temp["product_id"] = val_scored_dlinear_full_temp["product_id"].astype(str)

    # Ensure dlinear_predictions_full_df also has string IDs for merging
    dlinear_predictions_full_df["store_id"] = dlinear_predictions_full_df["store_id"].astype(str)
    dlinear_predictions_full_df["product_id"] = dlinear_predictions_full_df["product_id"].astype(str)

    # Merge only the dlinear_prediction column
    val_scored_dlinear_full_temp = val_scored_dlinear_full_temp.merge(
        dlinear_predictions_full_df[["store_id", "product_id", "day_idx", "dlinear_prediction"]],
        on=["store_id", "product_id", "day_idx"],
        how="left"
    )

    # Evaluate DLinear only on rows where predictions are available
    val_scored_dlinear_full_filtered = val_scored_dlinear_full_temp.dropna(subset=['dlinear_prediction']).copy()

    result_dlinear_full = evaluate_forecast(
        val_scored_dlinear_full_filtered,
        pred_col="dlinear_prediction"
    )

    print("\n--- DLinear Evaluation (Full Dataset) ---")
    display(result_dlinear_full)
else:
    print("No series processed for DLinear on the full dataset, potentially due to insufficient data.")

#### D1 Comparison Analysis

Note: Select the better forcasting model for D2 task which is LightGBM

###7b. D2 - Recovering and Forecasting

1. Create sample of 5000 serries
2. Base line in sample data with LightGBM and unrecovered demand
3. Recover the censored data in sample and forecast with LightGBM
4. Compare WAPE
5.

In [ ]:
# Sample of 5000 series

num_series_to_sample = 5000

unique_series_ids = history['series_id'].unique()
if len(unique_series_ids) > num_series_to_sample:
    # Đảm bảo rng đã được khởi tạo (nếu chưa, hãy thêm dòng: rng = np.random.default_rng(RANDOM_SEED))
    sampled_series_ids = rng.choice(unique_series_ids, size=num_series_to_sample, replace=False)
    history_d2_sampled = history[history['series_id'].isin(sampled_series_ids)].copy()

    # Tùy chọn: Đặt lại `series_id` để chúng tuần tự từ 1 đến `num_series_to_sample`
    series_id_map_d2 = {old_id: new_id for new_id, old_id in enumerate(sampled_series_ids, 1)}
    history_d2_sampled['series_id'] = history_d2_sampled['series_id'].map(series_id_map_d2)

    print(f"\nĐã lấy mẫu {num_series_to_sample:,} series từ tổng số {len(unique_series_ids):,} series ban đầu cho tác vụ D2.")
    print(f"Hiện tại có {history_d2_sampled['series_id'].nunique():,} series trong dữ liệu D2 đã lấy mẫu.")
else:
    history_d2_sampled = history.copy() # Nếu số series ban đầu đã ít hơn, sử dụng toàn bộ dữ liệu
    print(f"\nSố lượng series ({len(unique_series_ids):,}) đã nhỏ hơn hoặc bằng {num_series_to_sample:,}. Không cần lấy mẫu cho D2.")

# Xác nhận kích thước mới và các series_id đã cập nhật
print(f"Tổng số hàng sau khi lấy mẫu cho D2: {len(history_d2_sampled):,}")
print(f"Các series_id duy nhất trong tập dữ liệu D2 đã lấy mẫu: {history_d2_sampled['series_id'].nunique()}")

# Cũng tạo val_d2_sampled từ val ban đầu để phù hợp với quy trình D2
# val_d2_sampled sẽ chỉ chứa các series_id có trong history_d2_sampled
val_d2_sampled = val[val['series_id'].isin(sampled_series_ids)].copy()
val_d2_sampled['series_id'] = val_d2_sampled['series_id'].map(series_id_map_d2)
print(f"Tổng số hàng trong tập validation D2 đã lấy mẫu: {len(val_d2_sampled):,}")

In [ ]:
# LGBMRegressor with sample data and unrecovered demand ( baseline)
from lightgbm import LGBMRegressor
# drop missing value
train_Lgbm_D2 = history_d2_sampled.dropna()
val_Lgbm_D2 = val_d2_sampled.dropna()

features = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount']

target = "sale_amount"

model = LGBMRegressor(random_state=1)
model.fit(train_Lgbm_D2[features], train_Lgbm_D2[target])

val_Lgbm_D2["prediction"] = model.predict(val_Lgbm_D2[features])
val_Lgbm_D2["prediction"] = val_Lgbm_D2["prediction"].clip(lower=0)

evaluate_forecast(val_Lgbm_D2)

#### D2 - Random pool sampling recovered data and LightGBM

In [ ]:
# --- Simple recovery: random pool sampling on D2 sampled data ---
print("\n--- Triển khai phục hồi mẫu ngẫu nhiên (Simple Recovery) trên dữ liệu D2 đã lấy mẫu ---")

# Calculate visible_sum from the operating window sales for the sampled D2 data
visible_sum_d2_sampled = np.nansum(np.where(op_stock_d2_sampled == 0, op_sales_d2_sampled, 0), axis=1)

# Impute missing hourly cells using random pool sampling
imputed_random_pool_d2 = op_sales_masked_d2_sampled.copy()
imputed_count_random_pool_d2 = 0

for h in range(16):
    col = imputed_random_pool_d2[:, h]
    mask = np.isnan(col)
    n_miss = mask.sum()
    if n_miss > 0:
        pool = col[~mask]
        if len(pool) > 0:
            # Use rng for reproducibility if pool is not empty
            imputed_random_pool_d2[mask, h] = np.maximum(0, rng.choice(pool, size=n_miss, replace=True))
            imputed_count_random_pool_d2 += n_miss
        else:
            # Fallback if the pool is empty (all values for this hour are NaN), impute with 0
            imputed_random_pool_d2[mask, h] = 0
            imputed_count_random_pool_d2 += n_miss

# Rebuild corrected daily target
recovered_sum_random_pool_d2 = np.sum(imputed_random_pool_d2, axis=1) # Sum of the (now) fully imputed hourly sales

# outside_slice is sales outside operating hours, adjusted to be non-negative
outside_slice_random_pool_d2 = np.maximum(history_d2_sampled["sale_amount"].values.astype(np.float32) - visible_sum_d2_sampled, 0)
recovered_daily_random_pool_d2 = outside_slice_random_pool_d2 + recovered_sum_random_pool_d2

history_d2_sampled["recovered_daily_sales_random_pool"] = recovered_daily_random_pool_d2

print(f"Đã điền {imputed_count_random_pool_d2:,} ô hàng giờ bằng mẫu ngẫu nhiên trên dữ liệu D2 đã lấy mẫu.")
print(f"Doanh số trung bình thô (D2 đã lấy mẫu): {history_d2_sampled['sale_amount'].mean():.4f}")
print(f"Doanh số đã phục hồi trung bình (mẫu ngẫu nhiên, D2 đã lấy mẫu): {history_d2_sampled['recovered_daily_sales_random_pool'].mean():.4f}")

# Re-split with recovered target
train_r_random_pool, val_r_random_pool = time_split(history_d2_sampled, horizon=7)

# LGBMRegressor on random pool recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_random_pool and val_r_random_pool dataframes
train_Lgbm_random_pool = train_r_random_pool.dropna(subset=features + ["recovered_daily_sales_random_pool"]).copy()
val_Lgbm_random_pool = val_r_random_pool.dropna(subset=features + ["recovered_daily_sales_random_pool"]).copy()

target_lgbm_random_pool = "recovered_daily_sales_random_pool"

model_random_pool = LGBMRegressor(random_state=RANDOM_SEED)
model_random_pool.fit(train_Lgbm_random_pool[features], train_Lgbm_random_pool[target_lgbm_random_pool])

val_Lgbm_random_pool["prediction"] = model_random_pool.predict(
    val_Lgbm_random_pool[features]
).clip(min=0)

print("\n=== LGBMRegressor trên dữ liệu đã phục hồi bằng mẫu ngẫu nhiên (D2 đã lấy mẫu) ===")
# When evaluating, 'sale_amount' should be the actual target
val_Lgbm_random_pool_eval = val_Lgbm_random_pool.copy()
val_Lgbm_random_pool_eval["sale_amount"] = val_Lgbm_random_pool_eval[target_lgbm_random_pool]

random_pool_lgbm_results = evaluate_forecast(
    val_Lgbm_random_pool_eval
)
print(random_pool_lgbm_results)

# Kết hợp kết quả vào all_recovery_results_d2 để so sánh
if 'all_recovery_results_d2' in globals():
    temp_df = pd.DataFrame({
        "Simple Random Pool + LGBM (D2 Sampled)": random_pool_lgbm_results
    }).T
    all_recovery_results_d2 = pd.concat([all_recovery_results_d2, temp_df])
    print("\n=== So sánh phục hồi D2 đã lấy mẫu (cập nhật) ===")
    display(all_recovery_results_d2)
else:
    print("\n=== Kết quả phục hồi mẫu ngẫu nhiên D2 đã lấy mẫu ===")
    display(pd.DataFrame({"Simple Random Pool + LGBM (D2 Sampled)": random_pool_lgbm_results}).T)

#### D2 - Per-Series Hourly Mean Recovered Data and LightGBM forecasting

In [ ]:
print("--- Chuẩn bị dữ liệu giờ cho tập D2 đã lấy mẫu ---")
# Mở rộng dữ liệu giờ từ các cột list cho lịch sử đã lấy mẫu
hourly_sales_d2_sampled = np.stack(history_d2_sampled["hours_sale"].values)
hourly_stock_ds_d2_sampled = np.stack(history_d2_sampled["hours_stock_status"].values)

# Tập trung vào cửa sổ hoạt động h06..h21 (chỉ số 6..21, 16 giờ)
op_sales_d2_sampled = hourly_sales_d2_sampled[:, 6:22].astype(np.float32)
op_stock_d2_sampled = hourly_stock_ds_d2_sampled[:, 6:22].astype(np.float32)

# Đánh dấu các giờ bị censored
op_sales_masked_d2_sampled = np.where(op_stock_d2_sampled == 1, np.nan, op_sales_d2_sampled)

total_cells_d2 = op_sales_masked_d2_sampled.size
missing_cells_d2 = np.isnan(op_sales_masked_d2_sampled).sum()
print(f"Cửa sổ hoạt động (D2 đã lấy mẫu): {op_sales_masked_d2_sampled.shape[1]} giờ (h06-h21)")
print(f"Số ô bị thiếu hàng giờ (D2 đã lấy mẫu): {missing_cells_d2:,} / {total_cells_d2:,} ({missing_cells_d2/total_cells_d2:.1%})")



print("\n--- Triển khai phục hồi trung bình hàng giờ theo từng series (không rò rỉ dữ liệu) trên dữ liệu D2 đã lấy mẫu ---")

# Xác định ngày bắt đầu validation từ lịch sử D2 đã lấy mẫu cho dữ liệu huấn luyện
# Điều này đảm bảo rằng các giá trị trung bình chỉ được tính từ giai đoạn huấn luyện
val_start_pshm_calc_d2 = history_d2_sampled['day_idx'].max() - 7 + 1 # horizon là 7

# Lọc history_d2_sampled để tạo một tập chỉ huấn luyện cho việc tính toán giá trị trung bình
train_history_for_pshm_calc_d2 = history_d2_sampled[history_d2_sampled["day_idx"] < val_start_pshm_calc_d2].copy()

# Tính toán trung bình hàng giờ theo từng series CHỈ TỪ DỮ LIỆU HUẤN LUYỆN
per_series_hourly_mean_data_d2 = {}

train_series_ids_d2 = train_history_for_pshm_calc_d2['series_id'].unique()

for series_id in train_series_ids_d2:
    # Lấy các chỉ mục gốc của history_d2_sampled tương ứng với series_id này VÀ nằm trong tập huấn luyện
    original_index_labels_for_series_in_train_d2 = train_history_for_pshm_calc_d2[
        train_history_for_pshm_calc_d2['series_id'] == series_id
    ].index

    # Chuyển đổi các nhãn chỉ mục gốc này thành các chỉ mục vị trí 0 dựa trên
    # trong history_d2_sampled (và do đó là các mảng numpy op_sales_d2_sampled/op_stock_d2_sampled)
    positional_indices_for_series_in_train_d2 = history_d2_sampled.index.get_indexer(original_index_labels_for_series_in_train_d2)

    # Trích xuất dữ liệu bán hàng và trạng thái tồn kho của cửa sổ hoạt động cho series này
    # từ các mảng op_sales_d2_sampled/op_stock_d2_sampled ĐẦY ĐỦ, nhưng chỉ cho các hàng trong giai đoạn huấn luyện.
    series_op_sales_train_d2 = op_sales_d2_sampled[positional_indices_for_series_in_train_d2, :]
    series_op_stock_train_d2 = op_stock_d2_sampled[positional_indices_for_series_in_train_d2, :]

    # Che (mask) dữ liệu bán hàng cho các trường hợp hết hàng trong dữ liệu huấn luyện của series này
    series_op_sales_masked_train_d2 = np.where(series_op_stock_train_d2 == 1, np.nan, series_op_sales_train_d2)

    hourly_means_d2 = np.full(series_op_sales_masked_train_d2.shape[1], np.nan)

    for hour_idx in range(series_op_sales_masked_train_d2.shape[1]):
        hour_sales_data = series_op_sales_masked_train_d2[:, hour_idx]
        if not np.all(np.isnan(hour_sales_data)):
            hourly_means_d2[hour_idx] = np.nanmean(hour_sales_data)

    per_series_hourly_mean_data_d2[series_id] = hourly_means_d2

# Điền (impute) bằng trung bình hàng giờ theo từng series (áp dụng cho dữ liệu đã che đầy đủ bằng các giá trị trung bình từ huấn luyện)
imputed_pshm_d2 = op_sales_masked_d2_sampled.copy()

imputed_count_pshm_d2 = 0

for i in range(len(history_d2_sampled)): # Lặp qua từng quan sát series-ngày trong toàn bộ history_d2_sampled
    current_series_id = history_d2_sampled['series_id'].iloc[i]
    series_hourly_means = per_series_hourly_mean_data_d2.get(current_series_id)

    if series_hourly_means is not None: # Các giá trị trung bình tồn tại cho series này từ dữ liệu huấn luyện
        for h in range(16): # Lặp qua từng giờ trong cửa sổ hoạt động
            if np.isnan(imputed_pshm_d2[i, h]):
                mean_val = series_hourly_means[h]
                if not np.isnan(mean_val): # Chỉ điền nếu có giá trị trung bình cho series-giờ này
                    imputed_pshm_d2[i, h] = np.maximum(0, mean_val)
                    imputed_count_pshm_d2 += 1
                else:
                    # Dự phòng nếu giá trị trung bình của series-giờ cụ thể là NaN (ví dụ: series này luôn bị censored cho giờ này trong huấn luyện)
                    imputed_pshm_d2[i, h] = 0
    else:
        # Nếu một series_id nằm trong history_d2_sampled nhưng không nằm trong train_series_ids_d2 (ví dụ: nó chỉ xuất hiện trong validation),
        # hourly_means của nó sẽ không nằm trong per_series_hourly_mean_data_d2. Điền bằng 0 trong trường hợp này.
        for h in range(16):
            if np.isnan(imputed_pshm_d2[i, h]):
                imputed_pshm_d2[i, h] = 0


# Xây dựng lại target hàng ngày đã sửa (sử dụng imputed_pshm_d2 hiện bao gồm toàn bộ history_d2_sampled)
# visible_sum là tổng doanh số từ các giờ không bị censored (logic gốc)
visible_sum_pshm_d2 = np.nansum(np.where(op_stock_d2_sampled == 0, op_sales_d2_sampled, 0), axis=1)
recovered_sum_pshm_d2 = np.sum(imputed_pshm_d2, axis=1) # Tổng doanh số hàng giờ đã điền đầy đủ

# outside_slice là doanh số ngoài giờ hoạt động, được điều chỉnh để không âm
outside_slice_pshm_d2 = np.maximum(history_d2_sampled["sale_amount"].values.astype(np.float32) - visible_sum_pshm_d2, 0)
recovered_daily_pshm_d2 = outside_slice_pshm_d2 + recovered_sum_pshm_d2

history_d2_sampled["recovered_daily_sales_pshm"] = recovered_daily_pshm_d2

print(f"Đã điền {imputed_count_pshm_d2:,} ô hàng giờ bằng trung bình hàng giờ theo từng series (chỉ dữ liệu huấn luyện) trên dữ liệu D2 đã lấy mẫu.")
print(f"Doanh số trung bình thô (D2 đã lấy mẫu): {history_d2_sampled['sale_amount'].mean():.4f}")
print(f"Doanh số đã phục hồi trung bình (trung bình hàng giờ theo từng series, D2 đã lấy mẫu): {history_d2_sampled['recovered_daily_sales_pshm'].mean():.4f}")

# Chia lại với target đã phục hồi
train_r_pshm, val_r_pshm = time_split(history_d2_sampled, horizon=7)

# LGBMRegressor trên doanh số đã phục hồi trung bình hàng giờ theo từng series
from lightgbm import LGBMRegressor

# Drop các giá trị thiếu từ các dataframe train_r_pshm và val_r_pshm
train_Lgbm_pshm = train_r_pshm.dropna()
val_Lgbm_pshm = val_r_pshm.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_pshm = "recovered_daily_sales_pshm"

model_pshm = LGBMRegressor(random_state=RANDOM_SEED)
model_pshm.fit(train_Lgbm_pshm[features_lgbm], train_Lgbm_pshm[target_lgbm_pshm])

val_Lgbm_pshm["prediction"] = model_pshm.predict(val_Lgbm_pshm[features_lgbm])
val_Lgbm_pshm["prediction"] = val_Lgbm_pshm["prediction"].clip(lower=0)

print("\n=== LGBMRegressor trên dữ liệu đã phục hồi bằng trung bình hàng giờ theo từng series (D2 đã lấy mẫu) ===")
val_Lgbm_pshm_eval = val_Lgbm_pshm.copy()
val_Lgbm_pshm_eval["sale_amount"] = val_Lgbm_pshm_eval[target_lgbm_pshm] # Để evaluate_forecast sử dụng target chính xác

pshm_lgbm_results = evaluate_forecast(val_Lgbm_pshm_eval)
print(pshm_lgbm_results)

# Kết hợp kết quả vào một DataFrame để so sánh
all_recovery_results_d2 = pd.DataFrame({
    "Per-Series Hourly Mean + LGBM (D2 Sampled)": pshm_lgbm_results
}).T
print("\n=== So sánh phục hồi D2 đã lấy mẫu ===")
display(all_recovery_results_d2)

#### D2 - DLinear recovered data and LightGBM forecasting

In [ ]:
# Install first if needed
!pip install -q pypots statsmodels

import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMRegressor
from pypots.imputation import DLinear

In [ ]:
# =========================
# 1. Prepare 3D data
# =========================

num_series_unique = history_d2_sampled["series_id"].nunique()
num_days_unique = history_d2_sampled["day_idx"].nunique()
num_hours_op = op_sales_masked_d2_sampled.shape[1]  # 16 hours

series_id_to_idx = {
    sid: i for i, sid in enumerate(sorted(history_d2_sampled["series_id"].unique()))
}

day_idx_to_idx = {
    didx: i for i, didx in enumerate(sorted(history_d2_sampled["day_idx"].unique()))
}

# Keep NaN here. Do NOT fill missing values with 0.
X_sales_3d_nan = np.full(
    (num_series_unique, num_days_unique, num_hours_op),
    np.nan,
    dtype=np.float32
)

for row_pos, (_, row) in enumerate(history_d2_sampled.iterrows()):
    s_idx = series_id_to_idx[row["series_id"]]
    d_idx = day_idx_to_idx[row["day_idx"]]
    X_sales_3d_nan[s_idx, d_idx, :] = op_sales_masked_d2_sampled[row_pos, :]

In [ ]:
# =========================
# 2. Mask validation period to avoid leakage
# =========================

val_start_day_idx = val_d2_sampled["day_idx"].min()
train_days_count = day_idx_to_idx[val_start_day_idx]

X_train_for_imputation = X_sales_3d_nan.copy()

# Mask future validation days as NaN during fitting
X_train_for_imputation[:, train_days_count:, :] = np.nan

train_data_for_pypots_fit = {
    "X": X_train_for_imputation
}

# =========================
# 3. Train PyPOTS DLinear
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_dlinear_pypots = DLinear(
    n_steps=num_days_unique,
    n_features=num_hours_op,
    moving_avg_window_size=25,
    d_model=64,
    epochs=5,
    batch_size=32,
    device=device
)

model_dlinear_pypots.fit(train_data_for_pypots_fit)

# =========================
# 4. Impute full sampled data
# =========================

imputed_sales_dlinear_3d = model_dlinear_pypots.impute({
    "X": X_sales_3d_nan
})

# Convert to numpy if needed
if isinstance(imputed_sales_dlinear_3d, torch.Tensor):
    imputed_sales_dlinear_3d = imputed_sales_dlinear_3d.detach().cpu().numpy()

# =========================
# 5. Map 3D imputed data back to 2D hourly rows
# =========================

imputed_op_sales_dlinear_pypots = np.zeros_like(op_sales_masked_d2_sampled, dtype=np.float32)

for row_pos, (_, row) in enumerate(history_d2_sampled.iterrows()):
    s_idx = series_id_to_idx[row["series_id"]]
    d_idx = day_idx_to_idx[row["day_idx"]]
    imputed_op_sales_dlinear_pypots[row_pos, :] = imputed_sales_dlinear_3d[s_idx, d_idx, :]

imputed_op_sales_dlinear_pypots = np.maximum(0, imputed_op_sales_dlinear_pypots)


# =========================
# 6. Check whether imputation actually changed missing cells
# =========================

missing_mask = np.isnan(op_sales_masked_d2_sampled)

print("Missing ratio:", missing_mask.mean())

print("Mean imputed value on missing cells:",
      imputed_op_sales_dlinear_pypots[missing_mask].mean())

print("Min imputed value:",
      imputed_op_sales_dlinear_pypots[missing_mask].min())

print("Max imputed value:",
      imputed_op_sales_dlinear_pypots[missing_mask].max())


In [ ]:
# =========================
# 7. Rebuild recovered daily sales
# =========================

visible_sum_dlinear = np.nansum(
    np.where(op_stock_d2_sampled == 0, op_sales_d2_sampled, 0),
    axis=1
)

recovered_sum_dlinear = np.sum(imputed_op_sales_dlinear_pypots, axis=1)

outside_slice_dlinear = np.maximum(
    history_d2_sampled["sale_amount"].values.astype(np.float32) - visible_sum_dlinear,
    0
)

recovered_daily_dlinear = outside_slice_dlinear + recovered_sum_dlinear

history_d2_sampled["recovered_daily_sales_dlinear_pypots"] = recovered_daily_dlinear

# =========================
# 8. Check recovered target difference
# =========================

print("Mean raw sale_amount:",
      history_d2_sampled["sale_amount"].mean())

print("Mean recovered DLinear:",
      history_d2_sampled["recovered_daily_sales_dlinear_pypots"].mean())

print("Mean absolute difference:",
      (history_d2_sampled["recovered_daily_sales_dlinear_pypots"]
       - history_d2_sampled["sale_amount"]).abs().mean())

# =========================
# 9. Split after recovered target is created
# =========================

train_r_dlinear_pypots, val_r_dlinear_pypots = time_split(
    history_d2_sampled,
    horizon=7
)

In [ ]:
# =========================
# 10. Train LightGBM on recovered target
# =========================

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    "management_group_id",
    "city_id",
    "product_id",
    "holiday_flag",
    "is_censored",
    "discount"
]

target_lgbm = "recovered_daily_sales_dlinear_pypots"

train_lgbm = train_r_dlinear_pypots.dropna(subset=features_lgbm + [target_lgbm]).copy()
val_lgbm = val_r_dlinear_pypots.dropna(subset=features_lgbm + [target_lgbm]).copy()

model_lgbm_dlinear = LGBMRegressor(random_state=RANDOM_SEED)

model_lgbm_dlinear.fit(
    train_lgbm[features_lgbm],
    train_lgbm[target_lgbm]
)

val_lgbm["prediction"] = model_lgbm_dlinear.predict(
    val_lgbm[features_lgbm]
).clip(min=0)

# =========================
# 11. Evaluate with your function
# =========================

val_eval = val_lgbm.copy()

# Important: evaluate_forecast uses "sale_amount" as actual target.
# So replace sale_amount with recovered target for D2 evaluation.
val_eval["sale_amount"] = val_eval[target_lgbm]

dlinear_pypots_lgbm_results = evaluate_forecast(
    val_eval,
    pred_col="prediction"
)

print(dlinear_pypots_lgbm_results)

#### D2 Comparison Analysis

### 7c. Recommendation